In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
import torch
from typing import TypedDict

In [2]:
class LLMBlog(TypedDict):
    title:str
    outline:str
    blog:str

In [11]:
def outline_gen(state:LLMBlog)->LLMBlog:
    title = state['title']
    llm = ChatOllama(model="llama3.1:8b",temperature=0)
    device = torch.cuda.is_available()
    print(device)
    prompt = f"Generate an outline for a blog of title {title}"
    outline = llm.invoke(prompt).content
    state['outline'] = outline
    return state

In [12]:
def blog_gen(state:LLMBlog)->LLMBlog:
    title = state['title']
    outline = state['outline']
    llm = ChatOllama(model="llama3.1:8b",temperature=0)
    prompt = f"Generate a blog for the title {title} and outline {outline}"
    blog = llm.invoke(prompt).content
    state['blog']=blog
    return state

In [15]:
workflow = StateGraph(LLMBlog)

workflow.add_node('llm_outline',outline_gen)
workflow.add_node('llm_blog',blog_gen)

workflow.add_edge(START,'llm_outline')
workflow.add_edge('llm_outline','llm_blog')
workflow.add_edge('llm_blog',END)

graph = workflow.compile()

In [16]:
intial_state = {'title':'Rise of AI'}
result = graph.invoke(intial_state)
print(result)

False
{'title': 'Rise of AI', 'outline': 'Here is a suggested outline for a blog on the "Rise of AI":\n\n**I. Introduction**\n\n* Brief overview of Artificial Intelligence (AI) and its growing importance\n* Thesis statement: The rise of AI is transforming industries, revolutionizing the way we live and work, and raising important questions about the future of humanity.\n\n**II. What is AI?**\n\n* Definition of AI and its subfields (Machine Learning, Deep Learning, Natural Language Processing, etc.)\n* Explanation of how AI works, including algorithms, data, and computational power\n* Examples of AI applications in various industries\n\n**III. The Impact of AI on Industries**\n\n* Healthcare: AI-assisted diagnosis, personalized medicine, and robotic surgery\n* Finance: AI-powered trading, risk management, and customer service\n* Transportation: Self-driving cars, drones, and smart traffic systems\n* Education: AI-based learning platforms, adaptive assessments, and virtual teaching assis